In [21]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# read rich fields data
rich = pd.read_csv('',
                   dtype={
                       'cellphone_number' : 'str',
                       'id_number': 'str'
                   })

# standardize the primary identifying field
rich['umuzi_email'] = rich['umuzi_email'].str.lower()

In [23]:
mapper = {
    'Name': 'name',
    'Email': 'email',
    'Course': 'service_name',
    'Program Name': 'service_used',
    'Completion Time': 'date_service_accessed'
}

In [24]:
cohortMap = {
    'BE Green': 'BeGreen Cohort',
    'BE GREEN LB2':'BeGreen Cohort',
    'SAP Developer Associate C2': 'SAP_Umuzi_Y2',
    'SAP General Tech Consultant  C2': 'SAP_Umuzi_Y2',
    'SAP Human Capital Management C2': 'SAP_Umuzi_Y2',
    'VML (C50)': 'XH-AUXUI-Aug-25',
    'Capitec E9 - Java Development': 'e9_java',
    'WesBank  Project Management': 'SL-PM-Wesbank-Nov-25',
    'XPL Block 3 - Accelerating Success  - XPL_B3_AS': 'XA-Nov-25',
    'Capitec E10': 'e10_da',
    'Capitec E10 - Java': 'e10_da',
    'Experience Lab Advanced Web Development XA1': 'XH-AWD-Aug-25',
    'Advanced WD: XH-AWD-Feb26': 'XH-AWD-Feb-26',
    'Accenture: Skill Advancement': 'Accenture: Skill Advancement',
    'Year 3: Support Role(Generalist )': 'SAP_Umuzi_Y3_Jan26',
    'Year 3: Human Capital Management': 'SAP_Umuzi_Y3_Jan26',
    'Year 3: SAP Developer Associate': 'SAP_Umuzi_Y3_Jan26',
    'Capitec E10 - Java': 'SAP_Umuzi_Y3_Jan26',
    "BEGREEN LB 3: Business Acceleration": "BeGreen Cohort",
    "Foundational WebDevelopment: SL-FWD-Feb26 (C51_BBD)": "SL-FWD-Feb26",
    "Advanced Project management": "SAP Umuzi"
}

In [ ]:
directory_path = Path('')


files = directory_path.rglob('*.csv')

dataframes = [pd.read_csv(f.resolve()) for f in files]

df = (
    pd.concat(dataframes, ignore_index=False)
    .pipe(
        lambda df: df.loc[df['Completion Time'].notna()]
    )
    .pipe(
        lambda df : df[['Name', 'Email', 'Course', 'Program Name', 'Completion Time']]
    )
    .pipe(
        lambda x: x.rename(columns=mapper)
    )
    .assign(
        date_service_accessed = lambda df: df['date_service_accessed'].apply(lambda x: x + ':00' if len(x) == 16 else x)
    )
    .assign(
        
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed'], format="%Y-%m-%dT%H:%M:%S").dt.date,
        service_used = lambda df: df['service_used'].str.strip(),
        service_name = lambda df: df['service_name'].str.strip(),
        email=lambda df: df['email'].str.strip().str.lower()
    )
    .pipe(
        lambda df: (
            # df[df['date_service_accessed'] >= pd.to_datetime('2026-01-01').date()]
            df[df['date_service_accessed'] >= pd.to_datetime('2026-01-01').date()]
            .sort_values(by='email')
            .reset_index(drop=True)
            
        )
    )
)


In [27]:
df.head()

,name,email,service_name,service_used,date_service_accessed
0,Eddie Hlabangwane,1996eddiwardo@gmail.com,Foundations of Digital Marketing and E-commerce,BEGREEN LB 3: Business Acceleration,2026-02-22
1,Eddie Hlabangwane,1996eddiwardo@gmail.com,Attract and Engage Customers with Digital Mark...,BEGREEN LB 3: Business Acceleration,2026-02-24
2,Eddiwardo Hlabangwane,1996eddiwardo@gmail.com,SAP Professional Fundamentals,Year 3: Support Role(Generalist ),2026-02-19
3,Abigail Anyorikyo,abingunan@gmail.com,Attract and Engage Customers with Digital Mark...,BEGREEN LB 3: Business Acceleration,2026-02-28
4,Abigail Anyorikyo,abingunan@gmail.com,Foundations of Digital Marketing and E-commerce,BEGREEN LB 3: Business Acceleration,2026-02-19


In [28]:
df.loc[df['service_used']=='Cybersecurity Pathway']

,name,email,service_name,service_used,date_service_accessed
5,Elvis Frimpong Aboagye,aboagyeelvis89@gmail.com,Using Python to Access Web Data,Cybersecurity Pathway,2026-01-06
6,Elvis Frimpong Aboagye,aboagyeelvis89@gmail.com,Computer Networks and Network Security,Cybersecurity Pathway,2026-01-06
73,Anita Omuvwie,anitaomuvwie49@gmail.com,Foundations of Cybersecurity,Cybersecurity Pathway,2026-02-03
74,Anita Omuvwie,anitaomuvwie49@gmail.com,Accelerate Your Job Search with AI,Cybersecurity Pathway,2026-01-31
75,Anita Omuvwie,anitaomuvwie49@gmail.com,Tools of the Trade: Linux and SQL,Cybersecurity Pathway,2026-02-05
76,Anita Omuvwie,anitaomuvwie49@gmail.com,Play It Safe: Manage Security Risks,Cybersecurity Pathway,2026-02-04
77,Anita Omuvwie,anitaomuvwie49@gmail.com,"Assets, Threats, and Vulnerabilities",Cybersecurity Pathway,2026-02-07
78,Anita Omuvwie,anitaomuvwie49@gmail.com,Connect and Protect: Networks and Network Secu...,Cybersecurity Pathway,2026-02-06
419,Mokete Joy Modjela,joymodjela@gmail.com,Introduction to Cybersecurity Careers,Cybersecurity Pathway,2026-01-27
504,Ogunsakin Kolawole Solomon,kolawolesamson820@gmail.com,System Administration and IT Infrastructure Se...,Cybersecurity Pathway,2026-01-30


In [29]:
# Remap SAP Programs not in Retool
df.loc[df['service_used'].str.contains('Pathway', case=False), 'cohort_name'] = 'SAP Program(Not Listed on Retool)'

In [30]:
# Remap for Accenture skills
df.loc[df['service_used'].str.contains('Accenture', case=False), 'cohort_name'] = 'Accenture: Skill Advancement'

In [31]:
# check that all the cohorts/service used are in the cohort map
uniqueCohorts = set(df['service_used'])

# remove the SAP pathways
uniqueCohorts = uniqueCohorts - set(filter(lambda x: x.__contains__('Pathway'), uniqueCohorts))

missing = uniqueCohorts - set(cohortMap.keys())
assert missing == set(), f"The following are missing: \n{chr(10).join(sorted(missing))}"

In [32]:
df.loc[df['cohort_name'].isna(), 'cohort_name'] = df.loc[df['cohort_name'].isna(), 'service_used'].map(lambda x: cohortMap.get(x))

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1607 entries, 0 to 1606
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   name                   1607 non-null   object
 1   email                  1607 non-null   object
 2   service_name           1607 non-null   object
 3   service_used           1607 non-null   object
 4   date_service_accessed  1607 non-null   object
 5   cohort_name            1607 non-null   object
dtypes: object(6)
memory usage: 75.5+ KB


In [35]:
df.head()

,name,email,service_name,service_used,date_service_accessed,cohort_name
0,Eddie Hlabangwane,1996eddiwardo@gmail.com,Foundations of Digital Marketing and E-commerce,BEGREEN LB 3: Business Acceleration,2026-02-22,BeGreen Cohort
1,Eddie Hlabangwane,1996eddiwardo@gmail.com,Attract and Engage Customers with Digital Mark...,BEGREEN LB 3: Business Acceleration,2026-02-24,BeGreen Cohort
2,Eddiwardo Hlabangwane,1996eddiwardo@gmail.com,SAP Professional Fundamentals,Year 3: Support Role(Generalist ),2026-02-19,SAP_Umuzi_Y3_Jan26
3,Abigail Anyorikyo,abingunan@gmail.com,Attract and Engage Customers with Digital Mark...,BEGREEN LB 3: Business Acceleration,2026-02-28,BeGreen Cohort
4,Abigail Anyorikyo,abingunan@gmail.com,Foundations of Digital Marketing and E-commerce,BEGREEN LB 3: Business Acceleration,2026-02-19,BeGreen Cohort


## Enrich the Records

> records where the email exists in the enrichment dataset,
> but not as the primary identifier

In [38]:
# Emails present in the enrichment dataset but not as the primary (COALESCE’d) identifier.
# Likely personal emails that were overridden by an Umuzi email during canonical mapping.

b = (set(df['email']) - set(rich['umuzi_email'].str.lower())) # don't exist using primary identifier
a = set(df['email']) - set(rich['email'].str.lower()) # don't exist in the email field

b - a # missing from rich['umuzi_email'] but exist in rich['email']

set()

In [39]:
len(b)

88

> There are no south africans in Set B; dropping them. Relevant filtering fror the IDC process.

In [40]:
df.drop(
    labels = df.loc[df['email'].isin(b)].index,
    axis=0, inplace=True
)

In [41]:
premerge_rows = df.shape[0]

In [42]:
data = df.merge(
    left_on='email', right_on='umuzi_email',
    how='left', right=rich.drop(columns=['email'])
)

In [43]:
# make sure no new rows are introduced due to duplicates
assert premerge_rows == data.shape[0], f"Row mismatch; expected shape:{premerge_rows, data.shape[1]}, instead got {data.shape}"

In [44]:
data['learner_id'] = data['learner_id'].astype('int')
data['id_number'] = data['id_number'].str.rjust(13, '0')

In [45]:
data['age_service_accessed'] = (
    pd.to_datetime(data['date_service_accessed'])- pd.to_datetime(data['date_of_birth'])
).dt.days // 365

# add age bins
data['age_range'] = pd.cut(
    x=data['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)

In [46]:
data['month_of_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.month_name()

data['month_of_service_accessed'] = data['month_of_service_accessed'] + ' 2025'

In [47]:
data = data[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cohort_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

In [ ]:
# export to sink datasets
# data.to_csv('../Sink Datasets/indicator_7_coursera.csv', index=False)

In [ ]:
# data.to_csv('../Sink Datasets/indicator_7_data.csv', index=False, mode='a', header=False)